### Agent III: ReAct Agent - Practice & Exercises
In this notebook, you will practice building a ReAct (Reasoning and Acting) agent that uses tools, makes decisions with conditional routing, and executes actions.

#### Part 1: Follow-Along Practice
##### GoalBuild a ReAct agent that binds arithmetic tools (`add`, `subtract`, `multiply`) and dynamically decides whether to execute a tool or finish.

##### Steps:1. Define the state schema using `add_messages` to append messages to history.
2. Create tools and bind them to the LLM.
3. Define the LLM call node, the `should_continue` conditional router, and the `ToolNode` tool runner.
4. Build the graph connecting START -> agent, agent -> conditional edge (tools or END), and tools -> agent.

In [ ]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

load_dotenv()

# 1. Define State
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
# 2. Create tools
@tool
def add(a: int, b: int):
    """This is an addition function that adds 2 numbers together"""
    return a + b

@tool
def subtract(a: int, b: int):
    """Subtraction function"""
    return a - b

@tool
def multiply(a: int, b: int):
    """Multiplication function"""
    return a * b

tools = [add, subtract, multiply]
model = ChatOpenAI(model="gpt-4o").bind_tools(tools)

In [ ]:
# 3. Nodes and Routing logic
def model_call(state: AgentState) -> dict:
    sys = SystemMessage(content="You are an AI assistant. Use tools when needed.")
    response = model.invoke([sys] + list(state["messages"]))
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if not last_message.tool_calls:
        return "end"
    return "continue"

In [ ]:
# 4. Build and Compile Graph
graph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)
graph.add_node("tools", ToolNode(tools))

graph.set_entry_point("our_agent")
graph.add_conditional_edges(
    "our_agent",
    should_continue,
    {
        "continue": "tools",
        "end": END
    }
)
graph.add_edge("tools", "our_agent")
app = graph.compile()

In [ ]:
# Test it
for step in app.stream({"messages": [("user", "Add 40 and 12, then multiply by 6.")]}, stream_mode="values"):
    step["messages"][-1].pretty_print()

---

#### Part 2: Practice Exercises
##### Exercise 1: Division and Power ReAct Agent
###### GoalBuild a ReAct Agent that adds division (`divide`) and exponentiation (`power`) tools, and successfully handles mathematical queries requiring these functions.

###### Sample Input```python
{"messages": [("user", "Divide 100 by 4, and then raise the result to the power of 3.")]}
```

###### Sample OutputOutputs performing division, then exponentiation, returning the final answer: `15625.0`.

###### Plan1. **Define Tools:** Create two tools `@tool` for `divide(a: float, b: float)` and `power(base: float, exponent: float)`.
2. **Bind Tools:** Bind the new tools list to the model.
3. **Build Graph:** Set up the ReAct `StateGraph` with the agent node, the conditional router `should_continue`, and the tool node.
4. **Invoke:** Verify step-by-step tool execution.

In [ ]:
# TODO: Define your State, Tools, Nodes, Graph, Compile, and Invoke it here


<details>
<summary>💡 Click to see solution</summary>

```python
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

### 1. Stateclass AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

### 2. Define tools@tool
def divide(a: float, b: float) -> float:
    """Divide function that divides a by b."""
    return a / b

@tool
def power(base: float, exponent: float) -> float:
    """Power function that raises base to the power of exponent."""
    return base ** exponent

tools = [divide, power]
model = ChatOpenAI(model="gpt-4o").bind_tools(tools)

### 3. Nodes and Routerdef model_call(state: AgentState) -> dict:
    sys = SystemMessage(content="You are an AI assistant. Use tools when needed.")
    response = model.invoke([sys] + list(state["messages"]))
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if not last_message.tool_calls:
        return "end"
    return "continue"

### 4. Build Graphgraph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)
graph.add_node("tools", ToolNode(tools))
graph.set_entry_point("our_agent")
graph.add_conditional_edges(
    "our_agent",
    should_continue,
    {
        "continue": "tools",
        "end": END
    }
)
graph.add_edge("tools", "our_agent")
app = graph.compile()

### 5. Testfor step in app.stream({"messages": [("user", "Divide 100 by 4, and raise the result to the power of 3.")]}, stream_mode="values"):
    step["messages"][-1].pretty_print()
```
</details>